In [1]:
!pip install -q gradio matplotlib numpy

import gradio as gr
import matplotlib.pyplot as plt
import numpy as np

ROUGHNESS_LEVELS = {
    "Smooth Finish": {"ks_mm": 0.02, "cd_mult": 1.00, "scatter_mult": 1.00, "color": "#38bdf8"},
    "Light Roughness": {"ks_mm": 0.25, "cd_mult": 1.12, "scatter_mult": 1.15, "color": "#34d399"},
    "Moderate Roughness": {"ks_mm": 0.80, "cd_mult": 1.28, "scatter_mult": 1.38, "color": "#fbbf24"},
    "Heavy Roughness": {"ks_mm": 2.00, "cd_mult": 1.48, "scatter_mult": 1.70, "color": "#f87171"},
}


def simulate_surface_roughness(
    selected_levels, ref_length_cm, frontal_area_cm2, v_max_ms,
    aspect_ratio, oswald_e, cl_op, start_alt_m, base_sigma_pct, n_trials
):
    if not selected_levels:
        selected_levels = ["Smooth Finish", "Light Roughness", "Moderate Roughness", "Heavy Roughness"]

    rho = 1.225
    l_ref_m = max(ref_length_cm, 5.0) / 100.0
    a_ref_m2 = max(frontal_area_cm2, 5.0) / 10000.0
    v_arr = np.linspace(2.0, max(v_max_ms, 8.0), 160)
    n_runs = int(np.clip(n_trials, 50, 1000))

    # Schlichting-inspired roughness ratio contribution to zero-lift/parasitic drag
    base_cd0 = 0.028
    k_ind = 1.0 / (np.pi * max(aspect_ratio, 1.5) * np.clip(oswald_e, 0.45, 0.95))

    fig, (ax_drag, ax_cons) = plt.subplots(1, 2, figsize=(13.4, 4.9), dpi=130)
    fig.patch.set_facecolor("#0f172a")
    for ax in (ax_drag, ax_cons):
        ax.set_facecolor("#0f172a")
        ax.tick_params(colors="#94a3b8", labelsize=8)
        for spine in ax.spines.values():
            spine.set_color("#334155")

    rng = np.random.default_rng(101)
    summary_lines = []

    for level_name in selected_levels:
        cfg = ROUGHNESS_LEVELS[level_name]
        color = cfg["color"]
        ks_over_l = (cfg["ks_mm"] / 1000.0) / l_ref_m

        # Roughness increases skin-friction / parasitic drag coefficient
        rough_increment = 0.45 * (ks_over_l ** 0.45) * (cfg["cd_mult"] - 0.85)
        cd0_eff = np.maximum(0.005, base_cd0 * cfg["cd_mult"] + rough_increment)
        cd_tot = cd0_eff + k_ind * (cl_op ** 2)

        q_arr = 0.5 * rho * (v_arr ** 2)
        drag_n = q_arr * a_ref_m2 * cd_tot

        # Nominal L/D & glide range from start_alt_m
        ld_nom = cl_op / max(cd_tot, 1e-4)
        nominal_range_m = max(start_alt_m, 1.0) * ld_nom

        # Flight consistency across trials: higher roughness adds spread
        sigma_rel = (max(base_sigma_pct, 0.5) / 100.0) * cfg["scatter_mult"]
        trial_ranges_m = np.maximum(0.5, nominal_range_m * (1.0 + rng.normal(0.0, sigma_rel, n_runs)))

        mean_r = float(np.mean(trial_ranges_m))
        std_r = float(np.std(trial_ranges_m))
        consistency_pct = float(np.clip(100.0 * (1.0 - (std_r / max(mean_r, 1.0)) * 2.0), 0.0, 100.0))
        drag_at_max = float(drag_n[-1])

        ax_drag.plot(v_arr, drag_n, color=color, lw=2.0, label=f"{level_name} (Cd={cd_tot:.3f})")
        ax_cons.hist(
            trial_ranges_m, bins=22, alpha=0.52, color=color,
            edgecolor="#0f172a", label=f"{level_name} (±{std_r:.1f} m)"
        )
        ax_cons.axvline(mean_r, color=color, linestyle="--", lw=1.5)

        summary_lines.append(
            f"- **{level_name} (ks ≈ {cfg['ks_mm']:.2f} mm, ks/L ≈ {ks_over_l:.4f}):** "
            f"Cd ≈ {cd_tot:.3f}, Drag @ {v_max_ms:.0f} m/s ≈ {drag_at_max:.2f} N, "
            f"Mean Range ≈ {mean_r:.1f} m (± {std_r:.1f} m std), Consistency ≈ {consistency_pct:.0f}%."
        )

    ax_drag.set_title("1. Drag Force vs. Airspeed by Surface Roughness", color="#f1f5f9", fontsize=10, pad=6)
    ax_drag.set_xlabel("Airspeed, V (m/s)", color="#cbd5e1", fontsize=8.5)
    ax_drag.set_ylabel("Drag Force, F_D (N)", color="#cbd5e1", fontsize=8.5)
    ax_drag.legend(loc="upper left", fontsize=7.2, facecolor="#1e293b", edgecolor="#475569", labelcolor="#f8fafc")

    ax_cons.set_title("2. Flight Distance Spread & Consistency Across Trials", color="#f1f5f9", fontsize=10, pad=6)
    ax_cons.set_xlabel("Glide / Flight Distance (m)", color="#cbd5e1", fontsize=8.5)
    ax_cons.set_ylabel("Trial Count", color="#cbd5e1", fontsize=8.5)
    ax_cons.legend(loc="upper right", fontsize=7.2, facecolor="#1e293b", edgecolor="#475569", labelcolor="#f8fafc")

    fig.suptitle(
        f"Controlled Surface Roughness Study  |  L_ref={ref_length_cm:.0f} cm, A_ref={frontal_area_cm2:.0f} cm², Trials={n_runs}",
        color="#f8fafc", fontsize=11, fontweight="bold", y=0.98
    )
    plt.tight_layout(rect=[0, 0, 1, 0.94])

    md_doc = (
        "### 🧪 Surface Roughness Documentation Summary\n"
        + "\n".join(summary_lines)
        + "\n\n**Documented Findings:**\n"
        "- Increasing surface roughness raises effective drag coefficient (Cd) and drag force across airspeeds, "
        "reducing nominal aerodynamic efficiency (L/D) and mean glide range.\n"
        "- Higher roughness also widens trial-to-trial range spread (standard deviation), lowering flight consistency."
    )

    return fig, md_doc


with gr.Blocks() as demo:
    gr.Markdown("## 🧪 Surface Roughness Drag & Flight Consistency Simulator")

    with gr.Row():
        with gr.Column(scale=1):
            selected_levels = gr.CheckboxGroup(
                choices=list(ROUGHNESS_LEVELS.keys()),
                value=["Smooth Finish", "Light Roughness", "Moderate Roughness", "Heavy Roughness"],
                label="Select Surface Roughness Levels"
            )
            ref_length_cm = gr.Slider(10.0, 150.0, value=40.0, step=5.0, label="Model Reference Length, L_ref (cm)")
            frontal_area_cm2 = gr.Slider(10.0, 400.0, value=80.0, step=10.0, label="Model Reference Area (cm²)")
            v_max_ms = gr.Slider(8.0, 45.0, value=25.0, step=1.0, label="Max Airspeed, V (m/s)")
            aspect_ratio = gr.Slider(3.0, 14.0, value=7.5, step=0.5, label="Aspect Ratio (AR)")
            oswald_e = gr.Slider(0.55, 0.92, value=0.78, step=0.02, label="Oswald Efficiency Factor (e)")
            cl_op = gr.Slider(0.10, 1.10, value=0.50, step=0.05, label="Operating Lift Coefficient (CL)")
            start_alt_m = gr.Slider(2.0, 100.0, value=12.0, step=2.0, label="Reference Altitude, h0 (m)")
            base_sigma_pct = gr.Slider(1.0, 15.0, value=4.0, step=0.5, label="Baseline Trial Scatter (%)")
            n_trials = gr.Slider(100, 800, value=300, step=50, label="Monte Carlo Trials")

            run_btn = gr.Button("Run Roughness Study", variant="primary")

        with gr.Column(scale=2):
            rough_plot = gr.Plot(label="Drag & Flight Consistency Plots")
            rough_md = gr.Markdown()

    inputs = [
        selected_levels, ref_length_cm, frontal_area_cm2, v_max_ms,
        aspect_ratio, oswald_e, cl_op, start_alt_m, base_sigma_pct, n_trials
    ]
    run_btn.click(fn=simulate_surface_roughness, inputs=inputs, outputs=[rough_plot, rough_md])
    demo.load(fn=simulate_surface_roughness, inputs=inputs, outputs=[rough_plot, rough_md])

demo.launch(share=True, theme=gr.themes.Soft())

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://cf1969a6a04f391834.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
